# Module 03: Asymmetric Cryptography & PQC — Lab

This lab explores RSA, ECC, and post-quantum cryptographic primitives.

**Objectives:**
1. Implement basic RSA key generation and encryption/decryption
2. Perform elliptic curve point arithmetic
3. Understand PQC parameter sizes and compare with classical schemes

In [ ]:
from sympy import mod_inverse, isprime, nextprime, factorint
import hashlib
import secrets
import time

print("=" * 60)
print("RSA KEY GENERATION AND ENCRYPTION")
print("=" * 60)

def generate_rsa_keypair(bits=512):
    """Generate RSA key pair (small size for demo)"""
    p = nextprime(secrets.randbits(bits // 2))
    q = nextprime(secrets.randbits(bits // 2))
    while p == q:
        q = nextprime(secrets.randbits(bits // 2))
    
    n = p * q
    phi = (p - 1) * (q - 1)
    e = 65537
    d = mod_inverse(e, phi)
    
    return (n, e), (n, d), (p, q)

# Generate small RSA keys for demonstration
start = time.time()
pub, priv, primes = generate_rsa_keypair(bits=512)
gen_time = time.time() - start

print(f"Generated 512-bit RSA keys in {gen_time:.3f}s")
print(f"n = {pub[0]}")
print(f"e = {pub[1]}")
print(f"p = {primes[0]}")
print(f"q = {primes[1]}")

# Encrypt and decrypt
message = 42
ciphertext = pow(message, pub[1], pub[0])
decrypted = pow(ciphertext, priv[1], priv[0])

print(f"\nPlaintext:  {message}")
print(f"Ciphertext: {ciphertext}")
print(f"Decrypted:  {decrypted}")
print(f"Match: {'PASS' if decrypted == message else 'FAIL'}")

In [ ]:
# Elliptic Curve Point Arithmetic
print("=" * 60)
print("ELLIPTIC CURVE POINT ARITHMETIC (secp256k1-like)")
print("=" * 60)

# secp256k1 parameters: y^2 = x^3 + 7 (mod p)
p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
a = 0
b = 7
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
G = (Gx, Gy)
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

def ec_add(P, Q):
    """Point addition on secp256k1"""
    if P is None: return Q
    if Q is None: return P
    if P[0] == Q[0] and P[1] != Q[1]: return None  # Point at infinity
    
    if P == Q:
        lam = (3 * P[0] * P[0] * pow(2 * P[1], -1, p)) % p
    else:
        lam = ((Q[1] - P[1]) * pow(Q[0] - P[0], -1, p)) % p
    
    x3 = (lam * lam - P[0] - Q[0]) % p
    y3 = (lam * (P[0] - x3) - P[1]) % p
    return (x3, y3)

def ec_mul(k, P):
    """Scalar multiplication via double-and-add"""
    result = None
    addend = P
    while k > 0:
        if k & 1:
            result = ec_add(result, addend)
        addend = ec_add(addend, addend)
        k >>= 1
    return result

# Test point addition
G2 = ec_add(G, G)
G3 = ec_add(G2, G)
print(f"G  = ({G[0]:064x}, {G[1]:064x})")
print(f"2G = ({G2[0]:064x}, {G2[1]:064x})")
print(f"3G = ({G3[0]:064x}, {G3[1]:064x})")

# Verify nG = O (point at infinity)
nG = ec_mul(n, G)
print(f"\nnG = {nG} (should be None = point at infinity): {'PASS' if nG is None else 'FAIL'}")

# ECDH demonstration
print("\nECDH KEY EXCHANGE DEMONSTRATION")
print("-" * 40)
alice_priv = secrets.randbelow(n - 1) + 1
bob_priv = secrets.randbelow(n - 1) + 1
alice_pub = ec_mul(alice_priv, G)
bob_pub = ec_mul(bob_priv, G)

secret_alice = ec_mul(alice_priv, bob_pub)
secret_bob = ec_mul(bob_priv, alice_pub)

print(f"Alice private key: ...{alice_priv % (10**8):08d}")
print(f"Bob private key:   ...{bob_priv % (10**8):08d}")
print(f"Shared secret match: {'PASS' if secret_alice == secret_bob else 'FAIL'}")

In [ ]:
# PQC Parameter Comparison
print("=" * 60)
print("POST-QUANTUM vs CLASSICAL CRYPTOGRAPHY COMPARISON")
print("=" * 60)

schemes = [
    {"name": "RSA-2048", "type": "KEM/Sign", "pk": 256, "sk": 256, "ct": 256, "sig": 256,
     "security": 112, "quantum": "Broken"},
    {"name": "RSA-4096", "type": "KEM/Sign", "pk": 512, "sk": 512, "ct": 512, "sig": 512,
     "security": 140, "quantum": "Broken"},
    {"name": "ECC P-256", "type": "KEM/Sign", "pk": 64, "sk": 32, "ct": 64, "sig": 64,
     "security": 128, "quantum": "Broken"},
    {"name": "ML-KEM-512", "type": "KEM", "pk": 800, "sk": 1632, "ct": 768, "sig": 0,
     "security": 128, "quantum": "Secure"},
    {"name": "ML-KEM-768", "type": "KEM", "pk": 1184, "sk": 2400, "ct": 1088, "sig": 0,
     "security": 192, "quantum": "Secure"},
    {"name": "ML-KEM-1024", "type": "KEM", "pk": 1568, "sk": 3168, "ct": 1568, "sig": 0,
     "security": 256, "quantum": "Secure"},
    {"name": "ML-DSA-44", "type": "Sign", "pk": 1312, "sk": 2560, "ct": 0, "sig": 2420,
     "security": 128, "quantum": "Secure"},
    {"name": "ML-DSA-65", "type": "Sign", "pk": 1952, "sk": 4032, "ct": 0, "sig": 3293,
     "security": 192, "quantum": "Secure"},
    {"name": "SLH-DSA-128f", "type": "Sign", "pk": 32, "sk": 64, "ct": 0, "sig": 17088,
     "security": 128, "quantum": "Secure"},
]

print(f"{'Scheme':<16} {'Type':<10} {'PK(B)':<8} {'SK(B)':<8} {'CT/Sig(B)':<10} {'Sec(bits)':<10} {'PQC?'}")
print("-" * 80)
for s in schemes:
    ct_or_sig = s['ct'] if s['ct'] > 0 else s['sig']
    print(f"{s['name']:<16} {s['type']:<10} {s['pk']:<8} {s['sk']:<8} {ct_or_sig:<10} {s['security']:<10} {s['quantum']}")

print("\nKey Takeaway: PQC algorithms have larger keys/ciphertexts than ECC")
print("but remain quantum-secure. ML-KEM offers the best balance of")
print("security and bandwidth efficiency.")